# AELIONIX BLACKFORGE — Phase 9 Colab Validation

This notebook performs a deterministic, one-click validation of the **Network &
Infrastructure Security Capability Foundation**.

It exercises the full `blackforge.network` pipeline on the mock demo network:

* **host_discovery / port_discovery / service_observation** — reachability and
  open ports on the authorized `192.0.2.0/24` segment
* **protocol_identification / banner_observation / tls_observation** — bounded,
  credential-redacted protocol and banner evidence
* **dns_observation** — permissive DNS records (queries only)
* **network_exposure_analysis / infrastructure_modeling** — interface exposure
  and a non-cyclic topology (INFRASTRUCTURE/MEMBER_OF only)
* **service_application_correlation / network_evidence_collection** —
  application attribution and deterministic evidence assembly

Every capability runs the same guarded pipeline: request validation, scope /
authorization, **fail-closed port validation** (explicit integer lists bounded
to 1..65535), mock transport, normalization, evidence persistence, and
world-model materialization. **No free-form execution, no credential use, no
autonomous scanning beyond the scope, and no attack-graph edges.** Credential-
like fields are redacted from every banner, artifact, and observation raw blob.

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.

---

In [ ]:
import sys
import platform

print("Blackforge Phase 9 Colab Validation (Network & Infrastructure Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.network",
    "blackforge.network.models",
    "blackforge.network.transport",
    "blackforge.network.redaction",
    "blackforge.network.evidence",
    "blackforge.network.normalization",
    "blackforge.network.capabilities",
    "blackforge.network.materializer",
    "blackforge.network.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Network module imports: PASS")

---

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase9_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "authorization_ready",
            "model_router_ready"):
    assert verification[key], f"{key} must be True"
assert verification["network_ready"] is True, "network_ready must be True (11 typed capabilities)"
assert len(app.capability_registry.list_capabilities()) == 50

BOOTSTRAP_OK = app.healthy() and bool(verification["network_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (network_ready, 50 registered capabilities): PASS")

---

In [ ]:
from blackforge.network.capabilities import build_network_capabilities
from blackforge.network.models import NetworkMode, NetworkRequest
from blackforge.scope.models import TargetScope, Target, detect_target_type
from blackforge.core.types import RiskLevel, TargetType

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase9_net"
democ = [
    "192.0.2.0/24", "web.internal.example", "api.internal.example",
    "dns.internal.example", "mail.internal.example", "quiet.internal.example",
    "gateway.internal.example", "core-switch.internal.example",
    "firewall.internal.example",
]
EPOH = [
    "refused.internal.example", "slow.internal.example",
    "throttled.internal.example", "filtered.internal.example",
    "malformed.internal.example", "unauthorized.internal.example",
    "outofscope.internal.example", "missing.internal.example",
]
ALL = democ + EPOH
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in ALL],
    max_risk_level=RiskLevel.HIGH,
)
req = NetworkRequest(
    mission_id=MID, session_id="ses_phase9_net", scope=scope,
    mode=NetworkMode.ACTIVE, max_observations=500, timeout_seconds=30.0,
)

engine = app.network_engine
assert engine is not None and len(engine.capabilities) == 11
expected = sorted([
    "network.host_discovery",
    "network.port_discovery",
    "network.service_observation",
    "network.protocol_identification",
    "network.banner_observation",
    "network.dns_observation",
    "network.tls_observation",
    "network.network_exposure_analysis",
    "network.infrastructure_modeling",
    "network.service_application_correlation",
    "network.network_evidence_collection",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered network capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_medium = {"port_discovery", "service_observation", "protocol_identification",
           "banner_observation", "tls_observation", "network_evidence_collection"}
_active = {"port_discovery", "service_observation", "protocol_identification",
           "banner_observation", "tls_observation"}
_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    risk = meta.risk_level.value
    mode = meta.mode.value
    leaf = capability_id.split(".")[-1]
    assert (risk == "medium") == (leaf in _medium), capability_id
    assert (mode == "active") == (leaf in _active), capability_id
    assert meta.world_model, f"{capability_id} must materialize into the world model"
    print(
        f"  {meta.id:<38} risk={risk:<7} mode={mode:<8} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

---

In [ ]:
from blackforge.evidence.models import EvidenceRelation, EvidenceStatus
from blackforge.core.types import Confidence
from blackforge.world_model.query import RelationshipQuery, WorldQuery
from blackforge.world_model.models import EntityType, WorldLifecycle

# --- deterministic pipeline over every capability -------------------------
r_hosts = engine.discover_hosts(req, "192.0.2.0/24")
r_ports = engine.discover_ports(req, "web.internal.example", ports=[22, 80, 443])
r_serv = engine.observe_services(req, "web.internal.example", ports=[22, 80, 443])
r_proto = engine.identify_protocols(req, "web.internal.example", ports=[22, 443])
r_banner = engine.observe_banners(req, "web.internal.example", ports=[22, 80])
r_tls = engine.observe_tls(req, "web.internal.example", ports=[443])
r_exp = engine.analyze_exposure(req, "web.internal.example")
r_dns = engine.observe_dns(req, "dns.internal.example")
r_infra = engine.model_infrastructure(req, "192.0.2.0/24")
r_corr = engine.correlate_service_applications(req, "web.internal.example")
r_collect = engine.collect_network_evidence(req, "web.internal.example")

pipeline_runs = [r_hosts, r_ports, r_serv, r_proto, r_banner, r_tls, r_exp,
                 r_dns, r_infra, r_corr, r_collect]
from blackforge.network.models import NetworkStatus

statuses = []
for r in pipeline_runs:
    assert r.status == NetworkStatus.SUCCESS, (r.capability_id, r.status)
    statuses.append(f"{r.capability_id.split('.')[-1]}={len(r.observations)}")
print("All 11 network capabilities executed with status SUCCESS")
print("Observation counts:", " ".join(statuses))

assert r_hosts.observation_count == 5
assert r_ports.observation_count == 3  # 22, 80, 443 all open
assert r_serv.observation_count == 3   # ssh / http / https (nginx 1.24.0)
assert r_proto.observation_count == 3  # ssh / http / tls
assert r_tls.observation_count == 1
assert r_dns.observation_count == 8
assert r_infra.observation_count == 4
assert r_corr.observation_count == 3

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in pipeline_runs:
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 20, count_obs
print(f"DERIVED_FROM links: {count_obs} observations across {len(pipeline_runs)} artifacts")

# Every row persists as OBSERVED (network evidence never elevates).
stored = {e.id: e.status for e in app.evidence_store.list(limit=10000)}
assert all(stored[ev] == EvidenceStatus.OBSERVED for r in pipeline_runs
           for ev in r.evidence_ids), "network evidence must stay OBSERVED"
print("All network evidence rows persisted with status OBSERVED")

# --- world materialization --------------------------------------------------
wm = engine.world_model
entities = wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
etypes = {e.entity_type.value for e in entities}
needed = {"host", "port", "service", "protocol", "interface"}
assert needed <= etypes, etypes
assert "infrastructure" in etypes, "infrastructure topology must be materialized"
assert "application" in etypes, "service-application correlation must materialize"
print("World entity types:", ", ".join(sorted(etypes)))

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
need_rel = {"has_port", "runs_service", "uses_protocol", "has_interface",
            "member_of", "serves"}
assert need_rel <= rel_types, rel_types
attack_graph = rel_types & {"exploits", "can_compromise", "leads_to", "enables"}
assert not attack_graph, f"Attack-graph relationships must not be materialized: {attack_graph}"
print("Relationship types:", ", ".join(sorted(rel_types)))
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Host assertions carry banner/tls/network-evidence property keys.
host = wm.find_entity(MID, EntityType.HOST, "web.internal.example", namespace="192.0.2.10")
if host is None:
    host = next(h for h in entities if h.entity_type.value == "host"
                and h.namespace == "web.internal.example")
assert host is not None
assertions = wm.list_assertions(str(host.id), lifecycle=None)
prop_prefixes = {a.property_key.split(".")[0] for a in assertions}
assert {"banner", "banner_truncated", "tls", "tls_cert", "network_evidence"} <= prop_prefixes, prop_prefixes
print(f"Host {host.name} ({host.namespace}) assertions: {len(assertions)} "
      f"(prefixes {sorted(prop_prefixes)})")

# Confidence policy: active direct observations -> HIGH, derived -> MEDIUM.
_conf = {e.id: e.confidence for e in app.evidence_store.list(limit=10000)}
for ev_id in r_ports.evidence_ids[1:]:
    assert _conf[ev_id] == Confidence.HIGH, ev_id
for ev_id in r_corr.evidence_ids[1:]:
    assert _conf[ev_id] == Confidence.MEDIUM, ev_id
print("Confidence policy (active direct -> HIGH, derived -> MEDIUM): PASS")

---

In [ ]:
# Mission isolation: network work under a second mission is disjoint.
MID2 = "mission_phase9_net_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[_target("web.internal.example")],
    max_risk_level=RiskLevel.HIGH,
)
req2 = NetworkRequest(
    mission_id=MID2, session_id="ses_phase9_net_2", scope=scope2,
    mode=NetworkMode.ACTIVE, max_observations=500, timeout_seconds=30.0,
)
res2 = engine.discover_ports(req2, "web.internal.example", ports=[22])
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_ports.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert engine.world_model.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: banner + artifact + observation raw data never carry secrets.
raw = engine.observe_banners(req, "api.internal.example", ports=[8080])
banner = next(o for o in raw.observations if o.port == 8080)
assert "top-secret" not in banner.banner
assert "demo-token" not in banner.banner
assert "demo-key" not in banner.banner
print("API banner redacted (no access_token / api_key / credentials leaked): PASS")

from blackforge.evidence.models import EvidenceType

_rows = {e.id: e for e in app.evidence_store.list(limit=10000)}
for ev_id in raw.evidence_ids:
    row = _rows[ev_id]
    if row.evidence_type == EvidenceType.ARTIFACT:
        assert "top-secret" not in row.raw_data
        assert "demo-token" not in row.raw_data
        assert "demo-key" not in row.raw_data
    else:
        assert "top-secret" not in row.raw_data
        assert "credential_value_XXX" not in row.raw_data
print("Artifact + observation raw data redacted (recursive credential fields): PASS")

from blackforge.network.redaction import (
    credential_value_redacted,
    redact_credential_fields,
    redact_banner_text,
)

doc = {
    "service": "inventory_api",
    "access_token": "demo-token-123",
    "credentials": {"api_password": "top-secret"},
    "user": "alice",
}
clean = redact_credential_fields(doc)
assert clean["service"] == "inventory_api"
assert clean["access_token"] == credential_value_redacted()
assert clean["credentials"] == credential_value_redacted()
assert clean["user"] == "alice"
json_banner = ('{"service": "inventory_api", "access_token": "demo-token-123", '
               '"api_key": "demo-key-abc", "credentials": {"api_password": "top-secret"}}')
assert "demo-token" not in redact_banner_text(json_banner)
assert "top-secret" not in redact_banner_text(json_banner)
assert "inventory_api" in redact_banner_text(json_banner)
print("Redaction unit behavior (credential-like fields -> safe REDACTED marker): PASS")

# Idempotency: a repeated run over the same target reuses the same evidence rows.
before = app.evidence_store.count(MID)
engine.discover_ports(req, "web.internal.example", ports=[22, 80, 443])
after = app.evidence_store.count(MID)
assert after == before, (before, after)
print("Idempotent re-run (no duplicate evidence rows): PASS")

---

In [ ]:
from blackforge.core.errors import AuthorizationError, NetworkExecutionError
from blackforge.network.models import NetworkStatus

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = True
try:
    engine.discover_ports(req, "203.0.113.99", ports=[22])
    denied_out = False
except AuthorizationError:
    pass
assert denied_out
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unknown capability is rejected (no generic execution surface).
unknown_rejected = True
try:
    engine.run(req, "network.not_real", "web.internal.example")
    unknown_rejected = False
except NetworkExecutionError:
    pass
assert unknown_rejected
print("Unknown capability rejected (no generic execution surface): PASS")

# 3) Ports are fail-closed: explicit integer lists, bounded 1..65535.
def _expect_raise(ports):
    raised = True
    try:
        engine.discover_ports(req, "web.internal.example", ports=ports)
        raised = False
    except NetworkExecutionError:
        pass
    return raised

assert _expect_raise([]), "empty port list must be rejected"
assert _expect_raise([22, "80"]), "non-integer port must be rejected"
assert _expect_raise([70000]), "out-of-range port must be rejected"
assert _expect_raise([22] * 70000), "oversized port list must be rejected"
assert _expect_raise("22,80"), "non-list parameter must be rejected"
print("Fail-closed port validation (empty / non-int / out-of-range / oversized): PASS")

# 4) Failure states on the mock error hosts.
_error_map = {
    "refused.internal.example": NetworkStatus.REQUEST_FAILED,
    "slow.internal.example": NetworkStatus.TIMEOUT,
    "throttled.internal.example": NetworkStatus.RATE_LIMITED,
    "filtered.internal.example": NetworkStatus.FILTERED,
    "malformed.internal.example": NetworkStatus.MALFORMED_RESPONSE,
    "unauthorized.internal.example": NetworkStatus.UNAUTHORIZED,
    "outofscope.internal.example": NetworkStatus.REQUEST_FAILED,
    "missing.internal.example": NetworkStatus.REQUEST_FAILED,
}
for host, expected in _error_map.items():
    got = engine.discover_ports(req, host, ports=[22])
    assert got.status == expected, (host, got.status, expected)
print("Failure state mapping (8 error hosts) verified: PASS")
quiet = engine.discover_ports(req, "quiet.internal.example")
assert quiet.status == NetworkStatus.NO_EVIDENCE
assert any("no open or documented ports" in w for w in quiet.warnings)
print("Quiet host -> NO_EVIDENCE with warning: PASS")

# 5) Passive mode is LOW confidence; active direct remains HIGH.
from blackforge.core.types import Confidence

req_pas = NetworkRequest(
    mission_id=MID, session_id="ses_phase9_net_pas", scope=scope,
    mode=NetworkMode.PASSIVE, max_observations=500, timeout_seconds=30.0,
)
pas = engine.discover_hosts(req_pas, "192.0.2.0/24")
assert pas.mode == NetworkMode.PASSIVE
pas_ev = {e.id: e.confidence for e in app.evidence_store.list(limit=10000)}
assert all(pas_ev[ev] == Confidence.LOW for ev in pas.evidence_ids[1:]), "PASSIVE -> LOW"
print("Confidence mode policy (PASSIVE -> LOW, ACTIVE direct -> HIGH): PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == engine.world_model.count_entities(MID)
host_entity = fresh_wm.find_entity(MID, EntityType.HOST, "web.internal.example", namespace="192.0.2.10")
if host_entity is None:
    host_entity = next((e for e in fresh_wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
                        if e.entity_type.value == "host"
                        and e.namespace == "web.internal.example"), None)
persisted_host = host_entity is not None
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert persisted_ev and persisted_wm and persisted_host and rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_host

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "network" / "engine.py").exists(),
    "phase9_modules": bool(
        (REPO_DIR / "blackforge" / "network" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "network" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "network" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "network" / "redaction.py").exists()
        and (REPO_DIR / "blackforge" / "network" / "normalization.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_network_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "evidence_observed": bool(count_obs >= 20),
    "world_materialized": bool(needed <= etypes and "infrastructure" in etypes),
    "assertions_bound": len(assertions) > 0 and {"banner", "tls"} <= prop_prefixes,
    "confidence_policy": True,
    "no_attack_graph": not bool(attack_graph),
    "scope_authorization": denied_out,
    "unknown_capability_rejected": unknown_rejected,
    "ports_bounded": True,
    "redaction_boundary": True,
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in r_ports.evidence_ids})),
    "idempotent_runs": bool(after == before),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_network_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["ports_bounded"]
    and phase_checks["redaction_boundary"]
    and phase_checks["no_attack_graph"]
    and phase_checks["idempotent_runs"]
)

print()
print("=" * 60)
print("PHASE 9 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---